## Practice Exercises

### **Easy**

1. **Greeting decorator** — Write a decorator `@shout` that uppercases and prints `"!!!"` after whatever the wrapped function prints. Apply it to a `say_hi()` function.
2. **Banner decorator** — Write `@banner` that prints a line of `=` (length 40) before and after the wrapped function's output. Use `*args, **kwargs` so it works on any function.

### **Medium**

1. **`@timer`** — Write a decorator that measures and prints how long the wrapped function took to run (use `time.perf_counter()`), forwards `*args, **kwargs`, and returns the original's result. Use `functools.wraps`.

2. **`Temperature` with `@property`** — Build a class storing `_celsius`. Expose `celsius` as a property with a setter that rejects values below -273.15 (`ValueError`). Add a computed `fahrenheit` property (`c * 9/5 + 32`) with no setter.

### **Hard**

1. **`@retry`** — Write a decorator factory `@retry(times=3, delay=1)` that re-runs the function on exception up to `times`, sleeping `delay` seconds between attempts, and re-raises if all attempts fail. Preserve metadata with `functools.wraps`.
2. **`@log_calls` with stacking** — Write a `@log_calls` decorator that logs the function name and arguments before each call. Stack it with your `@timer` and verify the order of output, then swap the order and explain the difference.

---

<a id="ch4-challenge-project"></a>

## Challenge Project

### Mini Access-Control System + Validated `BankAccount`

Build a small system that combines decorators for authorization with a validated `@property`.

**Spec:**

1. **`@requires_role(role)`** — a decorator factory. It wraps a function and only runs it if the current user has the required role; otherwise it prints/raises `"Permission denied"`. Assume each protected function receives a `user` dict like `{"name": "ram", "role": "admin"}` as its first argument.

2. **`BankAccount` class:**
   - Store the balance in `_balance`.
   - Expose `balance` as a `@property` (getter).
   - Add a `balance` **setter** that rejects negative values with `ValueError("balance cannot be negative")`.
   - Methods `deposit(amount)` and `withdraw(amount)` that update `balance` through the property (so validation always applies). `withdraw` must reject amounts greater than the current balance.

3. **Wire it together:** a `transfer(user, src, dst, amount)` function decorated with `@requires_role("admin")` that moves money between two `BankAccount`s. Test it with both an admin user (succeeds) and a regular user (denied).

**Hints:**
- The decorator factory has three layers: `def requires_role(role): def decorator(func): def wrapper(user, *args, **kwargs): ... return wrapper; return decorator`.
- In `wrapper`, check `if user.get("role") != role:` before calling `func`.
- In `withdraw`, set `self.balance = self.balance - amount` so the setter's validation fires automatically.
- Use `@functools.wraps(func)` on the wrapper.

**Stretch goals:**
- Add an `@audit_log` decorator that records every transfer, and stack it with `@requires_role`.
- Add a computed `is_overdrawn` property.
- Build a tiny route registry: a `routes = {}` dict and a `@route("/path")` decorator that registers handler functions into it — a mini Flask.



### **Easy**

1. **Greeting decorator** — Write a decorator `@shout` that uppercases and prints `"!!!"` after whatever the wrapped function prints. Apply it to a `say_hi()` function.
2. **Banner decorator** — Write `@banner` that prints a line of `=` (length 40) before and after the wrapped function's output. Use `*args, **kwargs` so it works on any function.

In [ ]:
''' 1. **Greeting decorator** — Write a decorator `@shout` that uppercases and prints `"!!!"` after 
whatever the wrapped function prints. Apply it to a `say_hi()` function.'''

def shout(func):
    def wrapper():
        func()
        print("!!!")
    return wrapper

@shout
def say_hi():
    print("Hi")

say_hi()

Hi
!!!


In [2]:
''' 2. **Banner decorator** — Write `@banner` that prints a line of `=` (length 40) before and after the 
wrapped function's output. Use `*args, **kwargs` so it works on any function.'''

def banner(func):
    def wrapper(*args, **kwargs):
        print("=" * 40)
        func(*args, **kwargs)
        print("=" * 40)
    return wrapper

@banner
def greet(name):
    print(f"Hello {name}")

greet("Manoj")

Hello Manoj


### **Medium**

1. **`@timer`** — Write a decorator that measures and prints how long the wrapped function took to run (use `time.perf_counter()`), forwards `*args, **kwargs`, and returns the original's result. Use `functools.wraps`.

2. **`Temperature` with `@property`** — Build a class storing `_celsius`. Expose `celsius` as a property with a setter that rejects values below -273.15 (`ValueError`). Add a computed `fahrenheit` property (`c * 9/5 + 32`) with no setter.

In [4]:
''' 1. **`@timer`** — Write a decorator that measures and prints how long the wrapped function took to 
run (use `time.perf_counter()`), forwards `*args, **kwargs`, and returns the original's result. 
Use `functools.wraps`.'''

import time
from functools import wraps

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        func(*args, **kwargs)
        end = time.perf_counter()
        print(f"{func.__name__} tooks {end - start:.6f} seconds")
    return wrapper

@timer
def calculate():
    total = 0
    for i in range(100000):
        total += i

    return total

calculate()


calculate tooks 0.008025 seconds


In [1]:
''' 2. **`Temperature` with `@property`** — Build a class storing `_celsius`. Expose `celsius` as a 
property with a setter that rejects values below -273.15 (`ValueError`). Add a computed `fahrenheit`
property (`c * 9/5 + 32`) with no setter.'''

class Temperature:
    def __init__(self, celsius):
        self._celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("Temperature is below 273.15 celsius")
        else:
            self._celsius = value

    @property
    def fahrenheit(self):
        return self._celsius * 9/5 + 32

    
temp = Temperature(25)

print(temp.celsius)
print(temp.fahrenheit)

temp.celsius = -300

print(temp.celsius)
print(temp.fahrenheit)


25
77.0


ValueError: Temperature is below 273.15 celsius

### **Hard**

1. **`@retry`** — Write a decorator factory `@retry(times=3, delay=1)` that re-runs the function on exception up to `times`, sleeping `delay` seconds between attempts, and re-raises if all attempts fail. Preserve metadata with `functools.wraps`.
2. **`@log_calls` with stacking** — Write a `@log_calls` decorator that logs the function name and arguments before each call. Stack it with your `@timer` and verify the order of output, then swap the order and explain the difference.


In [2]:
''' 1. **`@retry`** — Write a decorator factory `@retry(times=3, delay=1)` that re-runs the function on 
exception up to `times`, sleeping `delay` seconds between attempts, and re-raises if all attempts 
fail. Preserve metadata with `functools.wraps`.'''

import time
from functools import wraps

def retry(times = 3,delay = 1):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times+1):
                try:
                    return func(*args, **kwargs)
                except:
                    print(f"Attempt {attempt} failed.")

                    if attempt == times:
                        raise

                    time.sleep(delay)
        return wrapper
    return decorator


counter = 0


@retry(times=3, delay=1)
def test_function():
    global counter

    counter += 1

    if counter < 3:
        raise ValueError("Something went wrong")

    return "Success!"


print(test_function())

Attempt 1 failed.
Attempt 2 failed.
Success!


In [5]:
''' 2. **`@log_calls` with stacking** — Write a `@log_calls` decorator that logs the function name 
and arguments before each call. Stack it with your `@timer` and verify the order of output, then 
swap the order and explain the difference.'''

import time
from functools import wraps

def log_calls(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"The function name is {func.__name__} and the args are {args} and kwargs are {kwargs}")
        return func(*args, **kwargs)
    return wrapper


def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()

        result = func(*args, **kwargs)

        end = time.perf_counter()

        print(f"The function {func.__name__} takes {end - start:.6f} seconds")

        return result
    return wrapper


@log_calls
@timer
def calculate():
    total = 0
    for i in range(100000):
        total += i

    return total


result = calculate()
print("Result:", result)

The function name is calculate and the args are () and kwargs are {}
The function calculate takes 0.009248 seconds
Result: 4999950000


In [ ]:
''' 2. **`@log_calls` with stacking** — Write a `@log_calls` decorator that logs the function name 
and arguments before each call. Stack it with your `@timer` and verify the order of output, then 
swap the order and explain the difference.'''

import time
from functools import wraps

def log_calls(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"The function name is {func.__name__} and the args are {args} and kwargs are {kwargs}")
        return func(*args, **kwargs)
    return wrapper


def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()

        result = func(*args, **kwargs)

        end = time.perf_counter()

        print(f"The function {func.__name__} takes {end - start:.6f} seconds")

        return result
    return wrapper


@timer
@log_calls
def calculate():
    total = 0
    for i in range(100000):
        total += i

    return total


result = calculate()
print("Result:", result)

The function name is calculate and the args are () and kwargs are {}
The function calculate takes 0.008882 seconds
Result: 4999950000


When @log_calls is placed above @timer, log_calls executes first and prints the function name and arguments before the timer starts. When @timer is placed above @log_calls, the timer starts first, so the time measured includes the execution of log_calls as well as the function. The execution times may differ slightly because of normal runtime variation.

## Challenge Project

### Mini Access-Control System + Validated `BankAccount`

Build a small system that combines decorators for authorization with a validated `@property`.

**Spec:**

1. **`@requires_role(role)`** — a decorator factory. It wraps a function and only runs it if the current user has the required role; otherwise it prints/raises `"Permission denied"`. Assume each protected function receives a `user` dict like `{"name": "ram", "role": "admin"}` as its first argument.

2. **`BankAccount` class:**
   - Store the balance in `_balance`.
   - Expose `balance` as a `@property` (getter).
   - Add a `balance` **setter** that rejects negative values with `ValueError("balance cannot be negative")`.
   - Methods `deposit(amount)` and `withdraw(amount)` that update `balance` through the property (so validation always applies). `withdraw` must reject amounts greater than the current balance.

3. **Wire it together:** a `transfer(user, src, dst, amount)` function decorated with `@requires_role("admin")` that moves money between two `BankAccount`s. Test it with both an admin user (succeeds) and a regular user (denied).

**Hints:**
- The decorator factory has three layers: `def requires_role(role): def decorator(func): def wrapper(user, *args, **kwargs): ... return wrapper; return decorator`.
- In `wrapper`, check `if user.get("role") != role:` before calling `func`.
- In `withdraw`, set `self.balance = self.balance - amount` so the setter's validation fires automatically.
- Use `@functools.wraps(func)` on the wrapper.

**Stretch goals:**
- Add an `@audit_log` decorator that records every transfer, and stack it with `@requires_role`.
- Add a computed `is_overdrawn` property.
- Build a tiny route registry: a `routes = {}` dict and a `@route("/path")` decorator that registers handler functions into it — a mini Flask.



In [4]:
from functools import wraps

def requires_roles(role):
    def decorator(func):
        @wraps(func)
        def wrapper(user, *args, **kwargs):
            if user.get("role") != role:
                raise PermissionError("Permission Denied")
            return func(user, *args, **kwargs)

        return wrapper

    return decorator

class BankAccount:
    def __init__(self, balance = 0):
        self._balance = 0
        self.balance = balance

    @property
    def balance(self):
        return self._balance

    @balance.setter
    def balance(self, value):
        if value < 0:
            raise ValueError("Balance cannot be negative.")

        self._balance = value

    def deposite(self, amount):
        self.balance = self.balance + amount

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError("Insufficient balance.")

        self.balance = self.balance - amount


@requires_roles("admin")
def transfer(user, src, dst, amount):
    src.withdraw(amount)
    dst.deposite(amount)

    print(f"Transfer successfull: {amount} transferred "
          f"from source account to destination account.")



admin_user = {
    "name": "ram",
    "role": "admin"
}

regular_user = {
    "name": "shyam",
    "role": "user"
}

account1 = BankAccount(1000)
account2 = BankAccount(500)


print("Before transfer:")
print("Account 1:", account1.balance)
print("Account 2:", account2.balance)


print("\nAdmin transfer:")
transfer(admin_user, account1, account2, 300)


print("\nAfter admin transfer:")
print("Account 1:", account1.balance)
print("Account 2:", account2.balance)


print("\nRegular user transfer:")
transfer(regular_user, account1, account2, 100)


Before transfer:
Account 1: 1000
Account 2: 500

Admin transfer:
Transfer successfull: 300 transferred from source account to destination account.

After admin transfer:
Account 1: 700
Account 2: 800

Regular user transfer:


PermissionError: Permission Denied